In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class reservoir_dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        if not root_dir.endswith('/'):
            root_dir += '/'
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len([f for f in os.listdir(self.root_dir) if f.startswith("data_") and f.endswith(".npz")])

    def __getitem__(self, index):
        input_list = []
        output_list = []

        if isinstance(index, int):
            index = [index]

        for idx in index:
            idx = str(idx).rjust(4, "0")
            file_path = self.root_dir + "data_" + idx + ".npz"
            
            with np.load(file_path) as data:
                porosity = data['porosity']
                perm_r = data['perm_r']
                perm_z = data['perm_z']
                pressure_buildup = data['pressure_buildup']
                gas_saturation = data['gas_saturation']
                
                inj_rate = data['inj_rate']
                temperature = data['temperature']
                depth = data['depth']
                Swi = data['Swi']
                lam = data['lam']
                perf_interval = data['perf_interval']

            #  QC & DATA TRACKING 
            # Extract the actual values that violate the [0.0, 1.0] physical bounds
            clipped_gas_vals = gas_saturation[(gas_saturation < 0.0) | (gas_saturation > 1.0)]
            clipped_por_vals = porosity[(porosity < 0.0) | (porosity > 1.0)]

            # Core script checks (NaNs / Infs / Permeability)
            nan_count = sum(np.isnan(arr).sum() for arr in [porosity, perm_r, perm_z, pressure_buildup, gas_saturation])
            inf_count = sum(np.isinf(arr).sum() for arr in [porosity, perm_r, perm_z, pressure_buildup, gas_saturation])

            if nan_count > 0 or inf_count > 0:
                print(f"[WARNING] data_{idx}.npz contains NaNs ({nan_count}) or Infs ({inf_count})!")
            if np.any(perm_r <= 0.0) or np.any(perm_z <= 0.0):
                print(f"[CRITICAL WARNING] data_{idx}.npz has negative or zero permeability!")

            # Execution of clipping
            gas_saturation = np.clip(gas_saturation, 0.0, 1.0)
            porosity = np.clip(porosity, 0.0, 1.0)

            #  DETAILED CLIP LOGGING 
            if len(clipped_gas_vals) > 0 or len(clipped_por_vals) > 0:
                log_msg = f"[QC Log] data_{idx}.npz items clipped:\n"
                if len(clipped_gas_vals) > 0:
                    log_msg += f"  -> Gas Saturation values: {clipped_gas_vals.tolist()}\n"
                if len(clipped_por_vals) > 0:
                    log_msg += f"  -> Porosity values:       {clipped_por_vals.tolist()}\n"
                print(log_msg.strip())
            else:
                print(f"[QC Log] data_{idx}.npz: Clear (0 clipped values).")

            #  Data Tensor Formatting 
            log_perm_r = np.log10(perm_r)
            log_perm_z = np.log10(perm_z)

            input_array = np.zeros((10, 96, 200))
            nz = porosity.shape[0]
            
            input_array[0, :nz, :] = porosity
            input_array[1, :nz, :] = log_perm_r  
            input_array[2, :nz, :] = log_perm_z  
            input_array[3, :, :] = np.ones((96, 200)) * inj_rate
            input_array[4, :, :] = np.ones((96, 200)) * temperature
            input_array[5, :, :] = np.ones((96, 200)) * depth
            input_array[6, :, :] = np.ones((96, 200)) * Swi
            input_array[7, :, :] = np.ones((96, 200)) * lam
            input_array[8, :, :] = np.ones((96, 200)) * perf_interval[0]
            input_array[9, :, :] = np.ones((96, 200)) * perf_interval[1]

            input_tensor = torch.from_numpy(input_array).float()

            output_array = np.zeros((2, 96, 200, 24))
            output_array[0, :nz, :, :] = pressure_buildup
            output_array[1, :nz, :, :] = gas_saturation

            output_tensor = torch.from_numpy(output_array).float()

            input_list.append(input_tensor)
            output_list.append(output_tensor)
            
        sample = torch.stack(input_list, dim=0), torch.stack(output_list, dim=0)

        if len(index) == 1:
            sample = (sample[0].squeeze(0), sample[1].squeeze(0))

        if self.transform:
            sample = self.transform(sample)

        return sample


In [ ]:
import os
import re
import numpy as np
import torch
from torch.utils.data import Dataset

#  STATISTICS CALCULATOR 

def compute_dataset_statistics(root_dir):
    """
    Scans the dataset directory to compute global statistics.
    """
    files = [
        os.path.join(root_dir, f) 
        for f in os.listdir(root_dir) 
        if re.match(r"^data_(\d{4})\.npz$", f) and ":" not in f
    ]
    
    if not files:
        raise FileNotFoundError(f"No valid 'data_xxxx.npz' files found in {root_dir}")

    por_list, perm_r_list, perm_z_list = [], [], []
    scalars = {k: [] for k in ['inj', 'temp', 'depth', 'swi', 'lam', 'perf0', 'perf1']}
    pressure_max = 0.0

    for file_path in files:
        with np.load(file_path) as data:
            nz = data['porosity'].shape[0]
            
            # Flatten and collect active domain rows for spatial properties
            por_list.append(np.clip(data['porosity'][:nz, :], 0.0, 1.0).flatten())
            perm_r_list.append(np.log10(data['perm_r'][:nz, :]).flatten())
            perm_z_list.append(np.log10(data['perm_z'][:nz, :]).flatten())
            
            # Track targets and experimental parameters
            pressure_max = max(pressure_max, float(np.max(data['pressure_buildup'])))
            scalars['inj'].append(data['inj_rate'])
            scalars['temp'].append(data['temperature'])
            scalars['depth'].append(data['depth'])
            scalars['swi'].append(data['Swi'])
            scalars['lam'].append(data['lam'])
            scalars['perf0'].append(data['perf_interval'][0])
            scalars['perf1'].append(data['perf_interval'][1])

    por_all = np.concatenate(por_list)
    pr_all = np.concatenate(perm_r_list)
    pz_all = np.concatenate(perm_z_list)

    return {
        'porosity_mean': np.mean(por_all),           'porosity_std': np.std(por_all),
        'log_perm_r_mean': np.mean(pr_all),         'log_perm_r_std': np.std(pr_all),
        'log_perm_z_mean': np.mean(pz_all),         'log_perm_z_std': np.std(pz_all),
        
        'inj_rate_mean': np.mean(scalars['inj']),     'inj_rate_std': np.std(scalars['inj']) + 1e-8,
        'temperature_mean': np.mean(scalars['temp']), 'temperature_std': np.std(scalars['temp']) + 1e-8,
        'depth_mean': np.mean(scalars['depth']),      'depth_std': np.std(scalars['depth']) + 1e-8,
        'Swi_mean': np.mean(scalars['swi']),          'Swi_std': np.std(scalars['swi']) + 1e-8,
        'lam_mean': np.mean(scalars['lam']),          'lam_std': np.std(scalars['lam']) + 1e-8,
        'perf_0_mean': np.mean(scalars['perf0']),     'perf_0_std': np.std(scalars['perf0']) + 1e-8,
        'perf_1_mean': np.mean(scalars['perf1']),     'perf_1_std': np.std(scalars['perf1']) + 1e-8,
        
        'pressure_max': pressure_max if pressure_max > 0 else 1.0
    }

# TORCH TRANSFORM FOR SCALING AND NORMALIZATION 

class NormalizeReservoirSample(object):
    """
    A PyTorch-compliant transform classand applies scaling and grid tensor formatting.
    """
    def __init__(self, stats):
        self.stats = stats

    def __call__(self, raw_data_dict):
        porosity = raw_data_dict['porosity']
        perm_r = raw_data_dict['perm_r']
        perm_z = raw_data_dict['perm_z']
        
        #  SPATIAL SCALING (Log-Transform & Z-Score) 
        porosity_scaled = (porosity - self.stats['porosity_mean']) / self.stats['porosity_std']
        log_perm_r_scaled = (np.log10(perm_r) - self.stats['log_perm_r_mean']) / self.stats['log_perm_r_std']
        log_perm_z_scaled = (np.log10(perm_z) - self.stats['log_perm_z_mean']) / self.stats['log_perm_z_std']
        
        #  SCALAR SCALING (Pre-Broadcast Normalization) 
        inj_rate_scaled = (raw_data_dict['inj_rate'] - self.stats['inj_rate_mean']) / self.stats['inj_rate_std']
        temp_scaled = (raw_data_dict['temperature'] - self.stats['temperature_mean']) / self.stats['temperature_std']
        depth_scaled = (raw_data_dict['depth'] - self.stats['depth_mean']) / self.stats['depth_std']
        Swi_scaled = (raw_data_dict['Swi'] - self.stats['Swi_mean']) / self.stats['Swi_std']
        lam_scaled = (raw_data_dict['lam'] - self.stats['lam_mean']) / self.stats['lam_std']
        perf_0_scaled = (raw_data_dict['perf_interval'][0] - self.stats['perf_0_mean']) / self.stats['perf_0_std']
        perf_1_scaled = (raw_data_dict['perf_interval'][1] - self.stats['perf_1_mean']) / self.stats['perf_1_std']

        #  DATA TENSOR FORMATTING & PADDING 
        input_array = np.zeros((10, 96, 200))
        nz = porosity.shape[0]
        
        input_array[0, :nz, :] = porosity_scaled
        input_array[1, :nz, :] = log_perm_r_scaled  
        input_array[2, :nz, :] = log_perm_z_scaled  
        
        input_array[3, :, :] = np.ones((96, 200)) * inj_rate_scaled
        input_array[4, :, :] = np.ones((96, 200)) * temp_scaled
        input_array[5, :, :] = np.ones((96, 200)) * depth_scaled
        input_array[6, :, :] = np.ones((96, 200)) * Swi_scaled
        input_array[7, :, :] = np.ones((96, 200)) * lam_scaled
        input_array[8, :, :] = np.ones((96, 200)) * perf_0_scaled
        input_array[9, :, :] = np.ones((96, 200)) * perf_1_scaled

        input_tensor = torch.from_numpy(input_array).float()

        #  TARGET SCALING 
        output_array = np.zeros((2, 96, 200, 24))
        output_array[0, :nz, :, :] = raw_data_dict['pressure_buildup'] / self.stats['pressure_max'] 
        output_array[1, :nz, :, :] = raw_data_dict['gas_saturation']  

        output_tensor = torch.from_numpy(output_array).float()

        return input_tensor, output_tensor

# RESERVOIR DATASET CLASS 

class reservoir_dataset(Dataset):
    """
    Custom PyTorch Dataset for parsing and validating multi-variable 
    reservoir grid simulations dynamically during model training.
    """
    def __init__(self, root_dir, transform=None):
        if not root_dir.endswith('/'):
            root_dir += '/'
        self.root_dir = root_dir
        self.transform = transform 

        # FIX: Scan and store the actual filenames that exist on disk
        self.valid_files = sorted([
            f for f in os.listdir(self.root_dir) 
            if re.match(r"^data_(\d{4})\.npz$", f) and ":" not in f
        ])

    def __len__(self):
        # FIX: Return the true count of discovered files
        return len(self.valid_files)

    def __getitem__(self, index):
        input_list, output_list = [], []
        if isinstance(index, int):
            index = [index]

        for idx in index:
            # FIX: Map PyTorch's sequential index to the actual filename string
            filename = self.valid_files[idx]
            file_path = os.path.join(self.root_dir, filename)
            
            with np.load(file_path) as data:
                porosity = data['porosity']
                gas_saturation = data['gas_saturation']
                perm_r = data['perm_r']
                perm_z = data['perm_z']

                # --- DATA INTEGRITY QC CHECK ---
                nan_count = sum(np.isnan(arr).sum() for arr in [porosity, perm_r, perm_z, data['pressure_buildup'], gas_saturation])
                inf_count = sum(np.isinf(arr).sum() for arr in [porosity, perm_r, perm_z, data['pressure_buildup'], gas_saturation])

                if nan_count > 0 or inf_count > 0:
                    print(f"[WARNING] {filename} contains NaNs ({nan_count}) or Infs ({inf_count})!")
                if np.any(perm_r <= 0.0) or np.any(perm_z <= 0.0):
                    print(f"[CRITICAL WARNING] {filename} has negative or zero permeability!")

                # In-place clipping bounds execution
                gas_saturation = np.clip(gas_saturation, 0.0, 1.0)
                porosity = np.clip(porosity, 0.0, 1.0)

                raw_sample = {
                    'porosity': porosity, 'perm_r': perm_r, 'perm_z': perm_z,
                    'pressure_buildup': data['pressure_buildup'], 'gas_saturation': gas_saturation,
                    'inj_rate': data['inj_rate'], 'temperature': data['temperature'], 'depth': data['depth'],
                    'Swi': data['Swi'], 'lam': data['lam'], 'perf_interval': data['perf_interval']
                }

                if self.transform:
                    input_tensor, output_tensor = self.transform(raw_sample)
                else:
                    input_tensor = torch.from_numpy(porosity).float()
                    output_tensor = torch.from_numpy(gas_saturation).float()

                input_list.append(input_tensor)
                output_list.append(output_tensor)
            
        sample = torch.stack(input_list, dim=0), torch.stack(output_list, dim=0)
        return (sample[0].squeeze(0), sample[1].squeeze(0)) if len(index) == 1 else sample